# Spin8 A100 public release

A lightweight, reproducible **ordinary Qwen3.8-27B → Spin8 GGUF → public Hugging Face release** workflow. It downloads verified small runtime assets instead of embedding them in this notebook.

| Item | Fixed value |
| --- | --- |
| Source | `Qwen/Qwen3.8-27B` at `37561816e66b5b9153b1b64c10968c18c1e6f4a0` |
| Runtime and release helpers | `zapabob/llama.cpp-spin8` at `3c5b2476ab10b5210f0858addcd4f98ab1394d75` |
| Public destination | **`zapabobouj/Qwen3.8-27B-Spin8-GGUF`** |
| Encoder | CUDA, FP32 candidate search; 8,192 groups, 17 scales, 1,048,576 tile elements; seed 81,237 |
| Weight editing | No mathematics delta, no supplied direction, no additional training |

Select an **A100** runtime before starting. The runner checks the actual GPU and executes a CUDA arithmetic check. It preserves the installed Torch/CUDA environment. Native build and verification gates retain the strict FP32 parity profile and unchanged tolerances.

The detached job builds, downloads, quantises, verifies the full model, and calls `prepare_release.py` to create the model card and exact upload manifest. **It never uploads.** The final cell has a separate public-upload switch.

This notebook does not attach to an existing live job or certify a completed 27B release. On 23 November 2026, an A100 80 GB build and source download had passed and the detached conversion was running; completed full-model quality and publication results were not yet available.


## 1. Choose local working storage

`/content` is temporary VM storage. Re-running against the same directory can reuse verified source files and tensor caches on the **same VM**. This notebook does not mount Drive or create an automatic backup. A new VM does not inherit this directory.

No token belongs in this cell or in its JSON configuration. Public source downloads do not require an HF token. The upload cell alone reads the Colab Secret named `HF_TOKEN`.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib, json, os, re, socket, subprocess, sys, tempfile, uuid

WORK_DIR = Path("/content/spin8-public-release").expanduser().resolve()
BUILD_JOBS = 4
PUBLIC_REPO = "zapabobouj/Qwen3.8-27B-Spin8-GGUF"
PUBLIC_COMMIT = "3c5b2476ab10b5210f0858addcd4f98ab1394d75"
SOURCE_REVISION = "37561816e66b5b9153b1b64c10968c18c1e6f4a0"
RAW_ROOT = f"https://raw.githubusercontent.com/zapabob/llama.cpp-spin8/{PUBLIC_COMMIT}/"
MANIFEST_SHA256 = "5566b8203cb11d6478fe80de710d2929d886fcb08a03f857ecac0829eb01582a"
PAYLOAD = WORK_DIR / "payload"
HELPERS = WORK_DIR / "helpers"
RUNNER = HELPERS / "run_release_job.py"
RELEASE_HELPER = HELPERS / "prepare_release.py"
TEMPLATE = WORK_DIR / "HF_MODEL_CARD_TEMPLATE.md"
CONFIG_PATH = WORK_DIR / "release-config.json"
ACTIVE_JOB = WORK_DIR / "active-job.json"

ASSETS = {
    "examples/spin8-colab/run_release_job.py": (RUNNER, 14699, "23ea6d16eced449995a76ad91d47a696a6d837e93670102018eaf1fa7e6d4ce8"),
    "examples/spin8-colab/prepare_release.py": (RELEASE_HELPER, 10928, "50d15915b992d95d0ee4635e99e6e34515000b1baaaecdc1111bd9a81af80566"),
    "examples/spin8-colab/render_model_card.py": (HELPERS / "render_model_card.py", 22035, "d2f43d587cb7d8c11526328b3b3d87b9a34201360456826cba0101a98753ffc8"),
    "docs/spin8/HF_MODEL_CARD_TEMPLATE.md": (TEMPLATE, 13277, "72965c7cf0f1bcf29a2ed49bbaeca07f1f2bc625cc54833f62edeb1487fff688"),
}
CONFIG = {
    "source_repo": "Qwen/Qwen3.8-27B", "source_revision": SOURCE_REVISION,
    "work_dir": str(WORK_DIR), "payload_dir": str(PAYLOAD),
    "output_name": "Qwen3.8-27B-Spin8.gguf", "device": "cuda",
    "math_delta": False, "math_alpha": 0.0, "direction_path": "",
    "seed": 81237, "scale_steps": 17, "chunk_groups": 8192,
    "tile_elements": 1048576, "max_materialized_bytes": 2 << 30,
    "max_workers": 2, "disk_reserve_gib": 8,
    "hf_repo": PUBLIC_REPO, "hf_private": False,
}

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()

def read_json(path):
    path = Path(path)
    if not path.is_file() or path.is_symlink() or path.stat().st_size > 64 << 20:
        raise RuntimeError(f"Missing or invalid JSON: {path}")
    value = json.loads(path.read_text())
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected a JSON object: {path}")
    return value

def atomic_json(path, value):
    path = Path(path)
    if path.is_symlink():
        raise RuntimeError(f"Refusing a symlink: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False) as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
        temporary = Path(stream.name)
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

print("Work directory:", WORK_DIR)
print("Public destination:", PUBLIC_REPO)
print("No model job or upload has been started.")


## 2. Fetch and verify the pinned runtime assets

The manifest is checked against the fixed SHA-256 below. Every payload entry is then checked for its exact size and SHA-256. Release helpers and the model-card template have separate fixed hashes. Existing matching files are reused; mismatched files cause a stop rather than silently replacing files used by a running job.


In [ ]:
import urllib.request

def checked_path(root, name):
    relative = PurePosixPath(name)
    if (not name or relative.is_absolute() or str(relative) != name
            or any(part in (".", "..") for part in relative.parts) or "\\" in name):
        raise RuntimeError("Invalid asset path")
    destination = root / relative
    if destination.is_symlink() or not destination.resolve().is_relative_to(root.resolve()):
        raise RuntimeError("Asset path escapes its directory")
    return destination

def check_file(path, size, digest):
    path = Path(path)
    if (not path.is_file() or path.is_symlink() or path.stat().st_size != size
            or sha256_file(path) != digest):
        raise RuntimeError(f"Asset size or SHA-256 differs: {path.name}")

def fetch_verified(remote, path, size, digest):
    path = Path(path)
    if path.exists() or path.is_symlink():
        check_file(path, size, digest)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        request = urllib.request.Request(RAW_ROOT + remote, headers={"User-Agent": "Spin8-Colab-Public-Release"})
        with urllib.request.urlopen(request, timeout=60) as response, tempfile.NamedTemporaryFile("wb", dir=path.parent, delete=False) as stream:
            temporary = Path(stream.name)
            total = 0
            while True:
                block = response.read(1 << 20)
                if not block:
                    break
                total += len(block)
                if total > size:
                    raise RuntimeError("Download exceeds its pinned size")
                stream.write(block)
        check_file(temporary, size, digest)
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

WORK_DIR.mkdir(parents=True, exist_ok=True)
fetch_verified("examples/spin8-colab/payload/MANIFEST.json", PAYLOAD / "MANIFEST.json", 3865, MANIFEST_SHA256)
manifest = read_json(PAYLOAD / "MANIFEST.json")
entries = manifest["files"]
if (manifest.get("schema") != "spin8-colab-payload-v2" or len(entries) != 17
        or len({entry["path"] for entry in entries}) != len(entries)):
    raise RuntimeError("Unexpected payload manifest")
for entry in entries:
    path = checked_path(PAYLOAD, entry["path"])
    fetch_verified("examples/spin8-colab/payload/" + entry["path"], path, entry["bytes"], entry["sha256"])
for remote, (path, size, digest) in ASSETS.items():
    fetch_verified(remote, path, size, digest)

def verify_assets():
    check_file(PAYLOAD / "MANIFEST.json", 3865, MANIFEST_SHA256)
    for entry in read_json(PAYLOAD / "MANIFEST.json")["files"]:
        check_file(checked_path(PAYLOAD, entry["path"]), entry["bytes"], entry["sha256"])
    for path, size, digest in ASSETS.values():
        check_file(path, size, digest)

verify_assets()
if CONFIG_PATH.exists():
    if read_json(CONFIG_PATH) != CONFIG:
        raise RuntimeError("Existing configuration differs. Keep an active job's inputs unchanged and choose another work directory.")
else:
    atomic_json(CONFIG_PATH, CONFIG)
print(f"Verified manifest, {len(entries)} payload files and {len(ASSETS)} release assets.")
print("Torch/CUDA have not been installed or replaced.")


## 3. Start the detached A100 job

The shipped cell source sets `RUN_JOB = False`. Change it to `True` and run this cell to start the expensive build and model workflow. **After invoking it, edit the source assignment back to `RUN_JOB = False`.** Resetting the Python variable does not change saved cell source: a later **Run all** executes any `True` assignment still saved there. The launcher records the VM boot ID, process ID and Linux process start time, then returns immediately. The runner also holds an exclusive work-directory lock.

Re-running this cell does not start another job while a matching runner or its current child stage is alive. A completed READY job is reused. After a failed or interrupted job, an explicit new start can reuse matching local caches; it does not restart or delete the runtime.

Notebook disconnection need not stop this detached process. A VM reset, deletion, service limit, process failure or termination can stop it. Detachment does not guarantee completion. The initial build may take several minutes; the status cell remains available during the job.


In [ ]:
import fcntl

RUN_JOB = False  # Shipped False; restore this source assignment after explicitly starting.

def boot_id():
    return Path("/proc/sys/kernel/random/boot_id").read_text().strip()

def process_info(pid):
    try:
        fields = Path(f"/proc/{int(pid)}/stat").read_text().rpartition(")")[2].split()
        return {"start_ticks": int(fields[19]), "state": fields[0]}
    except (OSError, ValueError, TypeError, IndexError):
        return None

def process_matches(pid, ticks, recorded_boot):
    info = process_info(pid)
    return (recorded_boot == boot_id() and ticks is not None and info is not None
            and info["start_ticks"] == ticks and info["state"] not in ("Z", "X"))

def current_job():
    if not ACTIVE_JOB.exists():
        return None, None
    launch = read_json(ACTIVE_JOB)
    directory = Path(launch["job_dir"]).resolve()
    if not directory.is_relative_to((WORK_DIR / "reports/jobs").resolve()):
        raise RuntimeError("Job receipt is outside the expected directory")
    status_path = directory / "status.json"
    record = read_json(status_path) if status_path.exists() else None
    return launch, record

def live_job(launch, record):
    if launch is None:
        return False
    if process_matches(launch.get("runner_pid"), launch.get("runner_start_ticks"), launch.get("boot_id")):
        return True
    if record:
        if process_matches(record.get("runner_pid"), record.get("runner_start_ticks"), record.get("boot_id")):
            return True
        for stage in record.get("stages", []):
            if process_matches(stage.get("child_pid"), stage.get("child_start_ticks"), record.get("boot_id")):
                return True
    return False

def start_release_job():
    verify_assets()
    if read_json(CONFIG_PATH) != CONFIG:
        raise RuntimeError("Configuration changed; reload the intended settings before starting")
    with (WORK_DIR / ".notebook-launch.lock").open("a+") as launcher_lock:
        fcntl.flock(launcher_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        launch, record = current_job()
        if (launch and record is None and launch.get("launch_state") != "SPAWN_FAILED"
                and (launch.get("runner_pid") is None or launch.get("runner_start_ticks") is None)):
            raise RuntimeError("Previous launch identity is incomplete. Inspect its existing receipt/log before retrying; no second runner was started.")
        if live_job(launch, record):
            print("A matching runner or child stage is still active. Use the status cell.")
            return
        if record and record.get("status") == "READY":
            print("This work directory already has a READY release. Review it below.")
            return
        with (WORK_DIR / ".spin8-release-job.lock").open("a+") as runner_lock:
            try:
                fcntl.flock(runner_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError("Another runner owns this work directory; no duplicate was started") from None
        job_dir = WORK_DIR / "reports/jobs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
        job_dir.mkdir(parents=True, exist_ok=False)
        launch = {"schema": "spin8-notebook-launch-v1", "started_utc": datetime.now(timezone.utc).isoformat(),
                  "job_dir": str(job_dir), "boot_id": boot_id(), "hostname": socket.gethostname(),
                  "config_sha256": sha256_file(CONFIG_PATH), "public_commit": PUBLIC_COMMIT,
                  "runner_pid": None, "runner_start_ticks": None, "launch_state": "STARTING"}
        atomic_json(ACTIVE_JOB, launch)
        environment = os.environ.copy()
        for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN",
                    "GGML_CUDA_SPIN8_VALIDATE_F32", "NVIDIA_TF32_OVERRIDE", "GGML_CUDA_CUBLAS_COMPUTE_TYPE"):
            environment.pop(key, None)
        environment.update(PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        command = [sys.executable, "-u", "-B", str(RUNNER), "--config", str(CONFIG_PATH),
                   "--release-helper", str(RELEASE_HELPER), "--template", str(TEMPLATE),
                   "--jobs", str(BUILD_JOBS), "--job-dir", str(job_dir)]
        try:
            with (job_dir / "launcher.log").open("xb") as log:
                process = subprocess.Popen(command, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
                                           env=environment, start_new_session=True)
        except Exception:
            launch["launch_state"] = "SPAWN_FAILED"
            atomic_json(ACTIVE_JOB, launch)
            raise
        info = process_info(process.pid)
        launch.update(runner_pid=process.pid, runner_start_ticks=info["start_ticks"] if info else None,
                      launch_state="STARTED")
        atomic_json(ACTIVE_JOB, launch)
        print("Detached runner PID:", process.pid)
        print("Receipt:", job_dir / "status.json")
        print("The runner checks the actual A100 before building or downloading model weights.")

if RUN_JOB:
    RUN_JOB = False
    start_release_job()
else:
    print("Not started. Set RUN_JOB = True and run this cell when ready.")


## 4. Inspect progress without occupying the kernel

Re-run this cell when needed. It shows the receipt, verified process identity and a short log tail. A missing live process with a RUNNING receipt means the job is no longer confirmed active; it does **not** mean the conversion completed. The cell never stops a process or starts another one.

If a reconnection clears Python variables, rerun cells 1–3 with `RUN_JOB = False` before checking status. Existing matching assets and configuration are reused.


In [ ]:
LOG_LINES = 18

def show_job_status():
    launch, record = current_job()
    if launch is None:
        print("No job has been launched from this work directory.")
        return
    is_live = live_job(launch, record)
    summary = {"status": record.get("status") if record else "STARTING_OR_START_FAILED",
               "launch_state": launch.get("launch_state"),
               "live_runner_or_stage": is_live, "same_vm_boot": launch.get("boot_id") == boot_id(),
               "runner_pid": (record or {}).get("runner_pid", launch.get("runner_pid")),
               "runner_start_ticks": (record or {}).get("runner_start_ticks", launch.get("runner_start_ticks")),
               "stage": record.get("current_stage") if record else None,
               "receipt": str(Path(launch["job_dir"]) / "status.json")}
    if record and record.get("gpu"):
        summary["gpu"] = record["gpu"]["name"]
    if record and record.get("status") == "RUNNING" and not is_live:
        summary["notice"] = "No matching live process. Inspect the log before explicitly retrying."
    print(json.dumps(summary, indent=2))
    directory = Path(launch["job_dir"])
    log = directory / "job.log"
    if not log.exists():
        log = directory / "launcher.log"
    if log.exists():
        with log.open("rb") as stream:
            stream.seek(max(0, log.stat().st_size - 24000))
            lines = stream.read().decode("utf-8", errors="replace").splitlines()
        text = "\n".join(lines[-max(1, min(int(LOG_LINES), 60)):])
        print(re.sub(r"\bhf_[A-Za-z0-9]{10,}\b", "[REDACTED]", text))

show_job_status()


## 5. Review the prepared GGUF, model card and upload manifest

The detached runner calls `prepare_release.py` **after** the build, source download, conversion and real-model verification pass. This preserves the existing publication gates and records the exact files to publish. READY means local release preparation passed; it does not mean anything has been uploaded.

The model card states the ordinary baseline settings and reports only matching evidence. This notebook does not run a mathematics benchmark or claim a capability improvement. The resulting custom Spin8 GGUF requires the supplied patched runtime.

This notebook publishes the **exact manifest recorded by the completed runner**. If a separate workflow regenerates the manifest, for example after adding benchmark results, review that new manifest and use `prepare_release.py check --manifest ...` and `prepare_release.py upload --manifest ...` directly. Preserve the original runner receipt; do not rewrite or bypass it to make this notebook accept a different manifest.


In [ ]:
from IPython.display import Markdown, display

def ready_manifest():
    verify_assets()
    launch, record = current_job()
    if not record or record.get("status") != "READY":
        return None
    if sha256_file(CONFIG_PATH) != launch["config_sha256"]:
        raise RuntimeError("Configuration differs from this completed job")
    path = Path(record["upload_manifest"]).resolve()
    if not path.is_relative_to((WORK_DIR / "release").resolve()):
        raise RuntimeError("Release manifest is outside the expected directory")
    if sha256_file(path) != record["upload_manifest_sha256"]:
        raise RuntimeError("Prepared manifest differs from the runner receipt")
    result = read_json(path)
    if (result.get("status") != "READY" or result.get("repo_id") != PUBLIC_REPO
            or result.get("private") is not False or result.get("model_sha256") != record["model"]["sha256"]):
        raise RuntimeError("Release destination or model identity differs")
    return path, result

prepared = ready_manifest()
if prepared is None:
    print("Not READY yet. Re-run this cell after the detached job completes.")
else:
    manifest_path, release = prepared
    print(json.dumps({"public_repository": release["repo_id"], "model_file": release["model_file"],
                      "model_sha256": release["model_sha256"], "files": len(release["files"]),
                      "total_bytes": release["total_bytes"], "manifest": str(manifest_path)}, indent=2))
    readme = next(entry for entry in release["files"] if entry["path_in_repo"] == "README.md")
    check_file(Path(readme["local_path"]), readme["bytes"], readme["sha256"])
    display(Markdown(Path(readme["local_path"]).read_text()))


## 6. Explicitly publish to Hugging Face

**Public destination: [`zapabobouj/Qwen3.8-27B-Spin8-GGUF`](https://huggingface.co/zapabobouj/Qwen3.8-27B-Spin8-GGUF).** Add a write-capable `HF_TOKEN` in Colab Secrets and allow this notebook to read it. Set `PUBLIC_UPLOAD = True` only when publishing the reviewed release.

The shipped cell source sets `PUBLIC_UPLOAD = False`. **After invoking an upload, edit the source assignment back to `PUBLIC_UPLOAD = False`.** An in-memory reset does not edit saved cell source, so a later **Run all** can invoke upload again if its source still says `True`. The helper rechecks every local file, creates a public model repository if needed, uploads only the prepared manifest, and verifies remote sizes and Git/LFS identities. An existing private repository causes a stop; its visibility is not silently changed.

Upload runs in the foreground and can take time. It receives the Secret only through its child-process environment. The token is not written to configuration, receipts or logs, and output is redacted before display. After a successful upload, `publication.json` records the immutable HF commit and verification result. The notebook checks that its manifest hash matches the reviewed manifest and that the local manifest has remained unchanged.


In [ ]:
PUBLIC_UPLOAD = False  # Shipped False; restore this source assignment after explicitly uploading.

def publish_prepared_release():
    import signal
    from google.colab import userdata
    prepared = ready_manifest()
    if prepared is None:
        raise RuntimeError("A complete matching READY release is required before upload")
    manifest_path, release = prepared
    reviewed_manifest_sha256 = sha256_file(manifest_path)
    _, runner_receipt = current_job()
    if not runner_receipt or reviewed_manifest_sha256 != runner_receipt.get("upload_manifest_sha256"):
        raise RuntimeError("Reviewed manifest differs from the original runner receipt")
    secret = userdata.get("HF_TOKEN")
    if not isinstance(secret, str) or not secret.strip():
        raise RuntimeError("Colab Secret HF_TOKEN is missing or empty")
    environment = os.environ.copy()
    for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):
        environment.pop(key, None)
    environment.update(HF_TOKEN=secret, PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1",
                       HF_HUB_DISABLE_PROGRESS_BARS="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
    process = None
    try:
        if sha256_file(manifest_path) != reviewed_manifest_sha256:
            raise RuntimeError("Prepared manifest changed before upload")
        command = [sys.executable, "-u", "-B", str(RELEASE_HELPER), "upload", "--manifest", str(manifest_path)]
        process = subprocess.Popen(command, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1,
                                   env=environment, start_new_session=True)
        for line in process.stdout:
            safe = re.sub(r"\bhf_[A-Za-z0-9]{10,}\b", "[REDACTED]", line.replace(secret, "[REDACTED]"))
            print(safe, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError("Upload or remote verification failed; inspect the redacted output and publication receipt")
        report = read_json(manifest_path.with_name("publication.json"))
        if (report.get("status") != "PUBLISHED" or report.get("remote_files_verified") is not True
                or report.get("repo") != PUBLIC_REPO or report.get("output_sha256") != release["model_sha256"]
                or report.get("manifest_sha256") != reviewed_manifest_sha256):
            raise RuntimeError("Publication receipt is incomplete or mismatched")
        if sha256_file(manifest_path) != reviewed_manifest_sha256:
            raise RuntimeError("Manifest changed during publication; inspect the committed publication receipt")
        print("Verified public release:", report["url"])
    except BaseException:
        if process is not None and process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait()
        raise
    finally:
        if process is not None and process.stdout is not None:
            process.stdout.close()
        environment.pop("HF_TOKEN", None)
        secret = None

if PUBLIC_UPLOAD:
    PUBLIC_UPLOAD = False
    publish_prepared_release()
else:
    print("No upload. Review the READY release, then explicitly set PUBLIC_UPLOAD = True.")
